In [ ]:
# ============================================
# 1. ЗАГРУЗКА НЕОБХОДИМЫХ ВЕРСИЙ БИБЛИОТЕК
# ============================================

!pip install ultralytics pyyaml
!pip install albumentations
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [1]:
# ============================================
# 2. ИМПОРТ НЕОБХОДИМЫХ БИБЛИОТЕК
# ============================================

import os
import numpy as np
import warnings
import yaml
import matplotlib.pyplot as plt

import cv2

from ultralytics import YOLO

import torch
from torchvision.ops import box_iou

from sklearn.metrics import precision_recall_curve, auc, roc_curve


warnings.filterwarnings("ignore")

In [2]:
# ============================================
# 3. ПРОВЕРКА АППАРАТНЫХ РЕСУРСОВ
# ============================================
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch версия: 2.7.1+cu118
CUDA доступна: True
GPU: NVIDIA GeForce RTX 4060 Ti
GPU память: 8.59 GB


In [3]:
# ============================================
# 3. ЗАДАНИЕ КЛЮЧЕВЫХ РЕПОЗЕТОРИЕВ
# ============================================

DATASET_PATH = 'models/dataset_augmented.yaml'
MODEL_640x640_PATH = "models/YOLOv8n_dif_pic_sizes/YOLOv8n_augmented_640x640/yolov8n.pt"
MODEL_560x560_PATH = "models/YOLOv8n_dif_pic_sizes/YOLOv8n_augmented_560x560/yolov8n.pt"
MODEL_480x480_PATH = "models/YOLOv8n_dif_pic_sizes/YOLOv8n_augmented_480x480/yolov8n.pt"
MODEL_400x400_PATH = "models/YOLOv8n_dif_pic_sizes/YOLOv8n_augmented_400x400/yolov8n.pt"
MODEL_FINETUNE_560x560_PATH = "models/YOLOv8n_dif_pic_sizes/YOLOv8n_finetune_560x560/yolov8n.pt"
MODEL_FINETUNE_480x480_PATH = "models/YOLOv8n_dif_pic_sizes/YOLOv8n_finetune_480x480/yolov8n.pt"
MODEL_FINETUNE_400x400_PATH = "models/YOLOv8n_dif_pic_sizes/YOLOv8n_finetune_400x400/yolov8n.pt"

In [4]:
# ============================================
# 5. ИНИЦИАЛИЗАЦИЯ МОДЕЛИ YOLOv8n
# ============================================

def create_model(model_path):
    model = YOLO(model_path)

    # Информация о модели ДО обучения
    #print("\n" + "="*50)
    #print("ИНФОРМАЦИЯ О КАСТОМНОЙ МОДЕЛИ:")
    #print("="*50)

    # Подсчитываем параметры вручную
    #total_params = sum(p.numel() for p in model.model.parameters())
    #trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

    #print(f"\nОбщее количество параметров: {total_params:,}")
    #print(f"Обучаемые параметры: {trainable_params:,}")
    #print(f"Размер модели (float32): {total_params * 4 / 1024 / 1024:.2f} MB")
    #print(f"Ожидаемый размер (int8): {total_params * 1 / 1024 / 1024:.2f} MB")

    return model

In [8]:
# ==================================================
# 6. КОНФИГУРАЦИЯ ОБУЧЕНИЯ ДЛЯ МАКСИМАЛЬНОЙ СКОРОСТИ
# ==================================================

def train_model(model_path, dataset_path, model_name='yolov8n_augmented', num_epochs=200, imgsz=640, optimizer='AdamW', lr0=0.001, lrf=0.000001):
    """Обучение с гиперпараметрами для скорости"""

    # Создаем модель
    model = create_model(model_path)

    # Критические гиперпараметры для скорости
    training_config = {
        'patience': 0,
        'name': model_name,
        'data': dataset_path,  # наш датасет
        'epochs': num_epochs,  # Больше эпох для маленькой модели
        'imgsz': imgsz,   # Обучаем на 192, инференс будет на 160
        'batch': 64,    # Максимальный батч для скорости
        'workers': 4,   # Параллельная загрузка
        'device': '0' if torch.cuda.is_available() else 'cpu',
        # Оптимизатор для быстрой сходимости
        'optimizer': optimizer,  # Более стабильный чем SGD
        'lr0': lr0,   # Меньше learning rate для стабильности
        'lrf': lrf,    # Финальный lr = 0.002 * 0.01 = 2e-5
        'momentum': 0.9,
        'weight_decay': 0.0001,  # Регуляризация
        # Аугментация - усиленная для переобучения маленькой модели
        'hsv_h': 0.015,  # Hue
        'hsv_s': 0.7,    # Saturation
        'hsv_v': 0.4,    # Value
        'degrees': 10.0, # Поворот
        'translate': 0.1,# Сдвиг
        'scale': 0.5,    # Масштаб
        'shear': 2.0,    # Наклон
        'perspective': 0.0005, # Перспектива
        'flipud': 0.0,   # Вертикальный флип (не нужно)
        'fliplr': 0.5,   # Горизонтальный флип
        'mosaic': 1.0,   # Мозаика - ВКЛЮЧИТЬ
        'mixup': 0.2,    # Mixup - для регуляризации
        'copy_paste': 0.2, # Копи-паст аугментация
        # Регуляризация против переобучения
        'dropout': 0.0,  # Dropout для регуляризации
        'label_smoothing': 0.1, # Сглаживание меток
        # Эвристики для скорости
        'nbs': 64,  # Nominal batch size
        'overlap_mask': False,  # Не использовать маски
        'single_cls': False,    # Многоклассовая
        'amp': True,  # Automatic Mixed Precision - УСКОРЕНИЕ 2x!
        # Сохранение
        'save_period': 10,  # Сохранять каждые 10 эпох
        'save_json': False, # Не сохранять JSON (экономия места)
        'cache': False,     # Не кэшировать изображения (экономия RAM)
        # Детекция
        'max_det': 50,      # Максимум детекций
        'conf': 0.001,      # Confidence threshold при обучении
        'iou': 0.7,         # IOU threshold
        'rect': True,       # Прямоугольное обучение для скорости
    }

    #Запуск обучения
    results = model.train(**training_config)

    return model, results


def model_finetune(model_path, dataset_path, num_epochs=200, imgsz=640, optimizer='AdamW', lr0=0.001, lrf=0.000001):
    """Дообучение с гиперпараметрами для скорости"""

    # Создаем модель
    model = create_model(model_path)

    training_config = {
        'resume': True,  # КЛЮЧЕВОЙ ПАРАМЕТР!
        'model': model_path,  # Путь к модели
        'patience': 0,

        'data': dataset_path,  # наш датасет
        'epochs': num_epochs,  # Больше эпох для маленькой модели
        'imgsz': imgsz,   # Обучаем на 192, инференс будет на 160
        'batch': 64,    # Максимальный батч для скорости
        'workers': 4,   # Параллельная загрузка
        'device': '0' if torch.cuda.is_available() else 'cpu',
        # Оптимизатор для быстрой сходимости
        'optimizer': optimizer,  # Более стабильный чем SGD
        'lr0': lr0,   # Меньше learning rate для стабильности
        'lrf': lrf,    # Финальный lr = 0.002 * 0.01 = 2e-5
        'momentum': 0.9,
        'weight_decay': 0.0001,  # Регуляризация
        # Аугментация - усиленная для переобучения маленькой модели
        'hsv_h': 0.015,  # Hue
        'hsv_s': 0.7,    # Saturation
        'hsv_v': 0.4,    # Value
        'degrees': 10.0, # Поворот
        'translate': 0.1,# Сдвиг
        'scale': 0.5,    # Масштаб
        'shear': 2.0,    # Наклон
        'perspective': 0.0005, # Перспектива
        'flipud': 0.0,   # Вертикальный флип (не нужно)
        'fliplr': 0.5,   # Горизонтальный флип
        'mosaic': 1.0,   # Мозаика - ВКЛЮЧИТЬ
        'mixup': 0.2,    # Mixup - для регуляризации
        'copy_paste': 0.2, # Копи-паст аугментация
        # Регуляризация против переобучения
        'dropout': 0.0,  # Dropout для регуляризации
        'label_smoothing': 0.1, # Сглаживание меток
        # Эвристики для скорости
        'nbs': 64,  # Nominal batch size
        'overlap_mask': False,  # Не использовать маски
        'single_cls': False,    # Многоклассовая
        'amp': True,  # Automatic Mixed Precision - УСКОРЕНИЕ 2x!
        # Сохранение
        'save_period': 10,  # Сохранять каждые 10 эпох
        'save_json': False, # Не сохранять JSON (экономия места)
        'cache': False,     # Не кэшировать изображения (экономия RAM)
        # Детекция
        'max_det': 50,      # Максимум детекций
        'conf': 0.001,      # Confidence threshold при обучении
        'iou': 0.7,         # IOU threshold
        'rect': True,       # Прямоугольное обучение для скорости
    }

    #Запуск обучения
    results = model.train(**training_config)

    return model, results

In [10]:
# =====================================================
# 7. ПЕРВИЧНОЕ ОБУЧЕНИЕ МОДЕЛИ ДЛЯ ИЗОБРАЖЕНИЙ 640x640
# =====================================================

train_model(model_path=MODEL_640x640_PATH,
            dataset_path=DATASET_PATH,
            model_name='yolov8n_augmented_640x640',
            num_epochs=100,
            imgsz=640,
            optimizer='AdamW',
            lr0=0.001,
            lrf=0.000001)

ERROR! Session/line number was not unique in database. History logging moved to new session 170


KeyboardInterrupt: 

In [7]:
# ============================================
# 8. ДООБУЧЕНИЕ МОДЕЛИ ДЛЯ ИЗОБРАЖЕНИЙ 640x640
# ============================================

model_finetune(model_path='models/YOLOv8n_dif_pic_sizes/YOLOv8n_augmented_640x640/weights/best.pt',
               dataset_path=DATASET_PATH,
               num_epochs=100,
               imgsz=640,
               optimizer='SGD',
               lr0=0.001,
               lrf=0.000001)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.239  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, crop_fraction=1.0, cutmix=0.0, data=models/YOLOv8n_augmented/dataset_augmented.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=False, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=True, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, label_smoothing=0.0, line_width=None, lr0=0.01, lrf=0.01, mask_rati

AssertionError: models\YOLOv8n_augmented\yolov8n.pt training to 500 epochs is finished, nothing to resume.
Start a new training without resuming, i.e. 'yolo train model=models\YOLOv8n_augmented\yolov8n.pt'

In [15]:
# ============================================
# 9. ОБУЧЕНИЕ МОДЕЛИ ДЛЯ ИЗОБРАЖЕНИЙ 560x560
# ============================================

train_model(model_path=MODEL_560x560_PATH,
            dataset_path=DATASET_PATH,
            model_name='yolov8n_augmented_560x560',
            num_epochs=100,
            imgsz=560,
            optimizer='AdamW',
            lr0=0.001,
            lrf=0.00001)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.239  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=models/dataset_augmented.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=560, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=1e-06, mask_ratio=4, max_det=50, mixup=0.2, mode=train, model=models/YOLO

KeyboardInterrupt: 

In [6]:
# ============================================
# 10. ДООБУЧЕНИЕ МОДЕЛИ ДЛЯ ИЗОБРАЖЕНИЙ 560x560
# ============================================

model_finetune(model_path='models/YOLOv8n_dif_pic_sizes/YOLOv8n_augmented_560x560/weights/best.pt',
               dataset_path=DATASET_PATH,
               num_epochs=100,
               imgsz=560,
               optimizer='AdamW',
               lr0=0.001,
               lrf=0.00001)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.239  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=models/dataset_augmented.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=560, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=1e-06, mask_ratio=4, max_det=50, mixup=0.0, mode=train, model=runs\detec

KeyboardInterrupt: 

In [7]:
# ============================================
# 11. ОБУЧЕНИЕ МОДЕЛИ ДЛЯ ИЗОБРАЖЕНИЙ 480x480
# ============================================

train_model(model_path=MODEL_480x480_PATH,
            dataset_path=DATASET_PATH,
            model_name='yolov8n_augmented_480x480',
            num_epochs=100,
            imgsz=480,
            optimizer='AdamW',
            lr0=0.001,
            lrf=0.00001)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.239  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=models/dataset_augmented.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=1e-05, mask_ratio=4, max_det=50, mixup=0.2, mode=train, model=models/YOL

KeyboardInterrupt: 

In [ ]:
# ============================================
# 12. ДООБУЧЕНИЕ МОДЕЛИ ДЛЯ ИЗОБРАЖЕНИЙ 480x480
# ============================================

model_finetune(model_path='models/YOLOv8n_dif_pic_sizes/YOLOv8n_augmented_480x480/weights/best.pt',
               dataset_path=DATASET_PATH,
               num_epochs=200,
               imgsz=480,
               optimizer='AdamW',
               lr0=0.001,
               lrf=0.00001)

In [6]:
# ============================================
# 13. ОБУЧЕНИЕ МОДЕЛИ ДЛЯ ИЗОБРАЖЕНИЙ 400x400
# ============================================

train_model(model_path=MODEL_400x400_PATH,
            dataset_path=DATASET_PATH,
            model_name='yolov8n_augmented_400x400',
            num_epochs=200,
            imgsz=400,
            optimizer='AdamW',
            lr0=0.001,
            lrf=0.00001)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.239  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=models/dataset_augmented.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=400, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=1e-05, mask_ratio=4, max_det=50, mixup=0.2, mode=train, model=models/YOL

KeyboardInterrupt: 

In [6]:
# ============================================
# 14. ДООБУЧЕНИЕ МОДЕЛИ ДЛЯ ИЗОБРАЖЕНИЙ 400x400
# ============================================

model_finetune(model_path='models/YOLOv8n_dif_pic_sizes/YOLOv8n_augmented_400x400/weights/best.pt',
               dataset_path=DATASET_PATH,
               num_epochs=200,
               imgsz=400,
               optimizer='AdamW',
               lr0=0.001,
               lrf=0.00001)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.239  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=models/dataset_augmented.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=400, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=1e-05, mask_ratio=4, max_det=50, mixup=0.0, mode=train, model=runs\detec

KeyboardInterrupt: 

In [11]:
# ======================================================
# 15. ДООБУЧЕНИЕ МОДЕЛИ 640x640 НА ИЗОБРАЖЕНИЯХ 400x400
# ======================================================

model_finetune(model_path='models/YOLOv8n_dif_pic_sizes/YOLOv8n_augmented_400x400/weights/last.pt',
               dataset_path=DATASET_PATH,
               num_epochs=100,
               imgsz=400,
               optimizer='AdamW',
               lr0=0.001,
               lrf=0.00001)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
WARNING 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.239  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=models/dataset_augmented.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=400, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=1e-05, mask_ratio=4, max_det=50, mixup=0.2, mode=train, model=models/YOL

KeyboardInterrupt: 